In [9]:
!rm -rf InkubaLM-Challenge
!git clone https://github.com/melissafasol/InkubaLM-Challenge.git
%cd InkubaLM-Challenge

Cloning into 'InkubaLM-Challenge'...
remote: Enumerating objects: 234, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 234 (delta 12), reused 14 (delta 3), pack-reused 202 (from 1)
Receiving objects: 100% (234/234), 963.60 KiB | 11.47 MiB/s, done.
Resolving deltas: 100% (139/139), done.
/content/InkubaLM-Challenge


In [10]:
!git checkout refactor-src-structure
!git pull

Branch 'refactor-src-structure' set up to track remote branch 'refactor-src-structure' from 'origin'.
Switched to a new branch 'refactor-src-structure'
Already up to date.


In [5]:
!pip install langdetect
!pip install datasets
!pip install transformers datasets peft trl accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=646defe400cd52c7e93987d981baa0d2cb5dec1d3edf954888e9024e1810aef9
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 13.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dep

In [34]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 1.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp311-cp311-linux_x86_64.whl size=4313506 sha256=0f0e81775c3fcf553004c68e442372c5e79287e38c7f0f5ce3ed07144e30ecbd
  Stored in directory: /root/.cache/pip/wheels/65/4f/35/5057db0249224e9ab55a513fa6b79451473ceb7713017823c3
Successfully built fasttext


In [6]:
import pandas as pd
from langdetect import detect
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, concatenate_datasets, Dataset, Value, DatasetDict

from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from peft import PeftModel, PeftConfig
from sklearn.model_selection import train_test_split


In [11]:
from utils import model_function, eval

In [151]:
import fasttext
import pandas as pd


def looks_like_mistranslation(src: str, tgt: str, length_factor: float = 2.0) -> bool:
    return len(src.split()) <= 10 and len(tgt.split()) > length_factor * len(src.split())

def flag_mt_issues(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add column to flag suspicious translations based only on length heuristic.
    """
    df = df.copy()
    df["flag_mistranslation"] = df.apply(
        lambda row: looks_like_mistranslation(row["inputs"], row["targets"]),
        axis=1)
    df["needs_review"] = df["flag_mistranslation"]
    return df

# -------- Bootstrap Examples -------- #

def create_simple_mt_examples(language: str) -> pd.DataFrame:
    """Return a few high-quality examples for English → target language MT."""
    examples = {
        "hausa": {
            "inputs": [
                "Good morning.",
                "Thank you very much.",
                "I am learning Hausa.",
                "What is your name?",
                "Where are you going?"
            ],
            "targets": [
                "Ina kwana.",
                "Na gode sosai.",
                "Ina koyon Hausa.",
                "Menene sunanka?",
                "Ina za ka je?"
            ]
        },
        "swahili": {
            "inputs": [
                "Good morning.",
                "Thank you very much.",
                "I am learning Swahili.",
                "What is your name?",
                "Where are you going?"
            ],
            "targets": [
                "Habari za asubuhi.",
                "Asante sana.",
                "Ninajifunza Kiswahili.",
                "Jina lako nani?",
                "Unaenda wapi?"
            ]
        }
    }
    data = examples[language.lower()]
    return pd.DataFrame({
        "ID": [f"bootstrap_{language}_{i}" for i in range(len(data["inputs"]))],
        "task": ["mmt"] * len(data["inputs"]),
        "langs": ["eng-" + language[:3]] * len(data["inputs"]),
        "data_source": ["synthetic"] * len(data["inputs"]),
        "instruction": ["translate the following from English into {}.".format(language.capitalize())] * len(data["inputs"]),
        "inputs": data["inputs"],
        "targets": data["targets"]
    })

# -------- Standardization + Save -------- #

def clean_mt_dataset(
    df: pd.DataFrame,
    target_lang: str,
    lang_code: str,
    standard_instruction: str,
    output_file: str,
    include_flagged_csv: bool = True
) -> pd.DataFrame:
    """Clean MT dataset and save improved version."""

    # Flag issues
    df = flag_mt_issues(df, target_lang_code=lang_code)

    # Standardize instruction
    df["instruction"] = standard_instruction

    # Filter clean rows
    clean_df = df[~df["needs_review"]].copy()

    # Add simple high-quality examples
    boot_df = create_simple_mt_examples(target_lang)
    final_df = pd.concat([clean_df, boot_df], ignore_index=True)

    # Save final dataset
    final_df.to_csv(output_file, index=False)
    print(f"✅ Cleaned dataset saved to '{output_file}' with {len(final_df)} rows.")

    if include_flagged_csv:
        flagged_path = output_file.replace(".csv", "_flagged.csv")
        df[df["needs_review"]].to_csv(flagged_path, index=False)
        print(f"⚠️ Flagged rows saved to '{flagged_path}' for manual review.")

    return final_df


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#os.environ["TOKENIZERS_PARALLELISM"] = "false"
import os
from huggingface_hub import login

try:
    from google.colab import userdata

    # Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
    # vars as appropriate for your system.
    # userdata.get("HF_TOKEN") indicates that the name of the token in the Colab env is HF_TOKEN
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except:
    os.environ["HF_TOKEN"] = "----"

login(token=os.environ["HF_TOKEN"])

token = os.environ["HF_TOKEN"]
if token == "----":
    print("⚠️ Warning: No Hugging Face token found. Some models may not load.")
else:
    login(token=token)

In [ ]:
model_name = "lelapa/InkubaLM-0.4B"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=os.environ["HF_TOKEN"])
model = model_function.load_model(model_name)

In [32]:
# Load the Swahili dataset from a local CSV file
mt_train = pd.read_parquet("hf://datasets/lelapa/MTTrain/data/train-00000-of-00001.parquet")


hau_dataset = Dataset.from_pandas(
    mt_train[mt_train['langs']=='eng-hau']
)
swa_dataset = Dataset.from_pandas(
    mt_train[mt_train['langs']=='eng-swa']
)

# If you need a DatasetDict to mimic the Hugging Face structure
dataset_dict = DatasetDict({
    "swahili": swa_dataset,
    "hausa": hau_dataset
})

# Print to verify
print(swa_dataset)
print(hau_dataset)

Dataset({
    features: ['ID', 'task', 'langs', 'data_source', 'instruction', 'inputs', 'targets', '__index_level_0__'],
    num_rows: 300
})
Dataset({
    features: ['ID', 'task', 'langs', 'data_source', 'instruction', 'inputs', 'targets', '__index_level_0__'],
    num_rows: 300
})


In [60]:
hau_df = mt_train.loc[mt_train['langs'] == 'eng-hau']

flagged_df = flag_mt_issues(hau_df)
flagged_df[["inputs", "targets", "flag_mistranslation", "needs_review"]].head()


,inputs,targets,flag_mistranslation,needs_review
1,this is possible because the on-demand applica...,wannan yana yiwuwa ne saboda sauƙi na gyare-gy...,False,False
2,"one trip took them to new orleans, a city with...","ɗaya daga cikin tafiya ya kai su new orleans, ...",False,False
7,it also has the history of jesus christ.,akwai kuma tarihin yesu kristi.,False,False
9,whatever mood you're in when you start watchin...,"komai inda za a fara farawa cikin slot din, za...",False,False
13,why we ought not to wed the pulpit with politi...,me ya sa ba za mu jefa kuri'a kan karin kudi g...,False,False


In [62]:
clean_df = flagged_df.loc[flagged_df['needs_review'] == False]

In [99]:
standard_instruction = "translate the following from English into Hausa."
clean_df["instruction"] = standard_instruction
clean_df["instruction"].unique()


<ipython-input-99-2e941860d758>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_df["instruction"] = standard_instruction


array(['translate the following from English into Hausa.'], dtype=object)

In [100]:
clean_df.head()

,ID,task,langs,data_source,instruction,inputs,targets
1,ID_5e3d2251_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from English into Hausa.,this is possible because the on-demand applica...,wannan yana yiwuwa ne saboda sauƙi na gyare-gy...
2,ID_fd263d98_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from English into Hausa.,"one trip took them to new orleans, a city with...","ɗaya daga cikin tafiya ya kai su new orleans, ..."
7,ID_96a1f4d4_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from English into Hausa.,it also has the history of jesus christ.,akwai kuma tarihin yesu kristi.
9,ID_50a2848b_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from English into Hausa.,whatever mood you're in when you start watchin...,"komai inda za a fara farawa cikin slot din, za..."
13,ID_034bfc55_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from English into Hausa.,why we ought not to wed the pulpit with politi...,me ya sa ba za mu jefa kuri'a kan karin kudi g...


In [67]:
boot_examples = create_simple_mt_examples("hausa")
boot_examples

,ID,task,langs,data_source,instruction,inputs,targets
0,bootstrap_hausa_0,mmt,eng-hau,synthetic,Translate the following from English into Hausa.,Good morning.,Ina kwana.
1,bootstrap_hausa_1,mmt,eng-hau,synthetic,Translate the following from English into Hausa.,Thank you very much.,Na gode sosai.
2,bootstrap_hausa_2,mmt,eng-hau,synthetic,Translate the following from English into Hausa.,I am learning Hausa.,Ina koyon Hausa.
3,bootstrap_hausa_3,mmt,eng-hau,synthetic,Translate the following from English into Hausa.,What is your name?,Menene sunanka?
4,bootstrap_hausa_4,mmt,eng-hau,synthetic,Translate the following from English into Hausa.,Where are you going?,Ina za ka je?


In [78]:
final_df = clean_df.drop(['flag_mistranslation', 'needs_review'], axis=1)

KeyError: "['flag_mistranslation', 'needs_review'] not found in axis"

In [84]:
final_df = pd.concat([clean_df, boot_examples], axis = 0)

In [85]:
final_df

,ID,task,langs,data_source,instruction,inputs,targets
1,ID_5e3d2251_dev_mt_eng-hau,mmt,eng-hau,wmt22,Translate the following from English into Hausa.,this is possible because the on-demand applica...,wannan yana yiwuwa ne saboda sauƙi na gyare-gy...
2,ID_fd263d98_dev_mt_eng-hau,mmt,eng-hau,wmt22,Translate the following from English into Hausa.,"one trip took them to new orleans, a city with...","ɗaya daga cikin tafiya ya kai su new orleans, ..."
7,ID_96a1f4d4_dev_mt_eng-hau,mmt,eng-hau,wmt22,Translate the following from English into Hausa.,it also has the history of jesus christ.,akwai kuma tarihin yesu kristi.
9,ID_50a2848b_dev_mt_eng-hau,mmt,eng-hau,wmt22,Translate the following from English into Hausa.,whatever mood you're in when you start watchin...,"komai inda za a fara farawa cikin slot din, za..."
13,ID_034bfc55_dev_mt_eng-hau,mmt,eng-hau,wmt22,Translate the following from English into Hausa.,why we ought not to wed the pulpit with politi...,me ya sa ba za mu jefa kuri'a kan karin kudi g...
...,...,...,...,...,...,...,...
0,bootstrap_hausa_0,mmt,eng-hau,synthetic,Translate the following from English into Hausa.,Good morning.,Ina kwana.
1,bootstrap_hausa_1,mmt,eng-hau,synthetic,Translate the following from English into Hausa.,Thank you very much.,Na gode sosai.
2,bootstrap_hausa_2,mmt,eng-hau,synthetic,Translate the following from English into Hausa.,I am learning Hausa.,Ina koyon Hausa.
3,bootstrap_hausa_3,mmt,eng-hau,synthetic,Translate the following from English into Hausa.,What is your name?,Menene sunanka?


Add more complex data

In [96]:
def create_bootstrap_mt_examples(language: str, level: str = "simple") -> pd.DataFrame:
    """
    Generate synthetic examples for MT training: 'simple', 'medium', or 'rich',
    and convert both inputs and targets to lowercase.
    """
    examples = {
        "hausa": {
            "simple": [
                ("Good morning.", "Ina kwana."),
                ("Thank you very much.", "Na gode sosai."),
                ("I am learning Hausa.", "Ina koyon Hausa."),
                ("What is your name?", "Menene sunanka?"),
                ("Where are you going?", "Ina za ka je?")
            ],
            "medium": [
                ("I will go to the market after breakfast.", "Zan je kasuwa bayan karin kumallo."),
                ("Do you know how to fix this problem?", "Ka san yadda za a gyara wannan matsalar?"),
                ("They decided to cancel the trip due to rain.", "Sun yanke shawarar soke tafiyar saboda ruwan sama."),
                ("The teacher asked us to submit the homework by Monday.", "Malam ya bukaci mu mika aikin gida kafin Litinin."),
                ("We are waiting for the doctor to arrive.", "Muna jiran likita ya iso.")
            ],
            "rich": [
                ("Despite the challenges, the committee agreed to implement the new policy.",
                 "Duk da ƙalubalen da ake fuskanta, kwamitin ya amince da aiwatar da sabon tsarin."),
                ("She explained the importance of education in rural communities.",
                 "Ta bayyana muhimmancin ilimi a cikin al'ummomin karkara."),
                ("The government launched a program to support young entrepreneurs.",
                 "Gwamnati ta ƙaddamar da shirin tallafawa matasan 'yan kasuwa."),
                ("It is essential to preserve cultural heritage for future generations.",
                 "Yana da matuƙar muhimmanci a kare gadon al'adu don amfanin zuri'a mai zuwa."),
                ("The organization provides healthcare services in underserved regions.",
                 "Kungiyar tana samar da ayyukan kiwon lafiya a yankuna da ba su da wadataccen sabis.")
            ]
        }
    }

    if language.lower() not in examples or level not in examples[language.lower()]:
        raise ValueError(f"No examples found for language='{language}' and level='{level}'.")

    level_data = examples[language.lower()][level]

    return pd.DataFrame({
        "ID": [f"bootstrap_{language}_{level}_{i}" for i in range(len(level_data))],
        "task": ["mmt"] * len(level_data),
        "langs": ["eng-" + language[:3]] * len(level_data),
        "data_source": ["synthetic"] * len(level_data),
        "instruction": [f"translate the following from english into {language.lower()}." for _ in level_data],
        "inputs": [pair[0].lower() for pair in level_data],
        "targets": [pair[1].lower() for pair in level_data]
    })


In [101]:
bootstrap_simple = create_bootstrap_mt_examples("hausa", level="simple")
bootstrap_medium = create_bootstrap_mt_examples("hausa", level="medium")
bootstrap_rich = create_bootstrap_mt_examples("hausa", level="rich")

bootstrap_all = pd.concat([bootstrap_simple, bootstrap_medium, bootstrap_rich], ignore_index=True)

final_df = pd.concat([clean_df,bootstrap_all], ignore_index=True)
#final_df.to_csv("hau_mt_augmented.csv", index=False)


In [109]:
from datasets import load_dataset

ds = load_dataset("mangaphd/hausaBERTdatatrain")

opus_df = pd.read_parquet("hf://datasets/Helsinki-NLP/opus-100/an-en/train-00000-of-00001.parquet")

In [112]:
opus_hausa_dataset = load_dataset("opus100", "en-ha")

In [114]:
train_data = opus_hausa_dataset["train"]
val_data = opus_hausa_dataset["validation"]
test_data = opus_hausa_dataset["test"]


In [118]:
train_data

Dataset({
    features: ['translation'],
    num_rows: 97983
})

In [130]:
# Select the first 100 examples correctly
subset = train_data.select(range(1000))

# Convert to a DataFrame
df_opus_train = pd.DataFrame({
    "inputs": [ex["translation"]["en"] for ex in subset],
    "targets": [ex["translation"]["ha"] for ex in subset]
})

df_opus_train["inputs"] = df_opus_train["inputs"].str.lower().str.strip()
df_opus_train["targets"] = df_opus_train["targets"].str.lower().str.strip()


In [131]:
df_opus_train = df_opus_train[
    df_opus_train["inputs"].notna() &
    df_opus_train["targets"].notna() &
    (df_opus_train["inputs"].str.strip().str.len() > 0) &
    (df_opus_train["targets"].str.strip().str.len() > 0)
]


In [132]:
import unicodedata

def normalize(text):
    return unicodedata.normalize("NFKC", text).lower().strip()

df_opus_train["inputs"] = df_opus_train["inputs"].apply(normalize)
df_opus_train["targets"] = df_opus_train["targets"].apply(normalize)

import re

def is_clean(text):
    return not bool(re.search(r'[{}[\]@#^*_<>~]', text))

df_opus_train = df_opus_train[
    df_opus_train["inputs"].apply(is_clean) &
    df_opus_train["targets"].apply(is_clean)
]



In [138]:
filter_keywords = ["o prophet", "nay,", "verily", "lo!", "messenger", "punishment", "behold", "god", "allah"]

def is_not_religious(text):
    return not any(kw in text for kw in filter_keywords)

df_opus_train = df_opus_train[
    df_opus_train["inputs"].apply(is_not_religious)
]


In [134]:
# Remove examples with fewer than 4 tokens in the Hausa target
df_opus_train = df_opus_train[
    df_opus_train["targets"].str.split().str.len() >= 4
]


In [149]:
df_opus_train = df_opus_train[df_opus_train["inputs"] != df_opus_train["targets"]]
df_opus_train= df_opus_train.drop_duplicates(subset=["inputs", "targets"])
MAX_INPUT_TOKENS = 128
MAX_TARGET_TOKENS = 128

df_opus_train = df_opus_train[
    (df_opus_train["inputs"].str.split().str.len() < MAX_INPUT_TOKENS) &
    (df_opus_train["targets"].str.split().str.len() < MAX_TARGET_TOKENS)
]

import string

def is_too_punctuated(text, threshold=0.3):
    punct_ratio = sum(1 for c in text if c in string.punctuation) / max(len(text), 1)
    return punct_ratio > threshold

df_opus_train = df_opus_train[~df_opus_train["inputs"].apply(is_too_punctuated)]

df_opus_train["inputs"] = df_opus_train["inputs"].str.replace("“|”", '"', regex=True)
df_opus_train["inputs"] = df_opus_train["inputs"].str.replace("’", "'", regex=True)


In [152]:
def format_opus_dataset(df: pd.DataFrame, language: str = "hausa") -> pd.DataFrame:
    """Format a cleaned OPUS dataset to match the synthetic MT format."""
    return pd.DataFrame({
        "ID": [f"opus_{language}_{i}" for i in range(len(df))],
        "task": ["mmt"] * len(df),
        "langs": ["eng-" + language[:3]] * len(df),
        "data_source": ["opus100"] * len(df),
        "instruction": [f"translate the following from English into {language.lower()}." for _ in range(len(df))],
        "inputs": df["inputs"].tolist(),
        "targets": df["targets"].tolist()
    })


In [156]:
df_opus_formatted = format_opus_dataset(df_opus_train, language="hausa")

In [163]:
from sklearn.utils import shuffle
test_dev = pd.concat([clean_df, df_opus_formatted], axis=0, ignore_index=True)
test_dev = shuffle(test_dev, random_state=42).reset_index(drop=True)

In [164]:
test_dev.head()

,ID,task,langs,data_source,instruction,inputs,targets
0,opus_hausa_418,mmt,eng-hau,opus100,translate the following from English into hausa.,"bring us our fathers, if what you say is true'","""sai ku zo da ubanninmu, idan kun kasance mãsu..."
1,opus_hausa_148,mmt,eng-hau,opus100,translate the following from English into hausa.,the first house (of worship) appointed for men...,"kuma lalle ne, ¦aki na farko da aka aza dõmin ..."
2,opus_hausa_233,mmt,eng-hau,opus100,translate the following from English into hausa.,so we exacted retribution from them: now see w...,"sabõda haka muka yi musu azãbar rãmuwa. to, ka..."
3,opus_hausa_429,mmt,eng-hau,opus100,translate the following from English into hausa.,"what, were we wearied by the first creation?","shin, to, mun kãsa ne game da halittar farko?"
4,ID_8f8fd13d_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from English into Hausa.,what's the symbolic meaning of the four animals?,ruhi ki ishi maa kya ab bane gi adi ki ishi maa?


In [180]:
import os
output_path = "/content/drive/MyDrive/InkubaLM-Challenge/Output/baseline_performance/datasets"
os.makedirs(output_path, exist_ok=True)

print(os.path.exists(output_path))  # Should return True
print(output_path)


True
/content/drive/MyDrive/InkubaLM-Challenge/Output/baseline_performance/datasets
['baseline_performance', 'new_datasets']


In [193]:
print("Paths equal:", output_path)
print("Exists:", os.path.exists(os.path.join(output_path, 'mt_hausa_dataset_supplement.csv')))


Paths equal: /content/drive/MyDrive/InkubaLM-Challenge/Output/baseline_performance/datasets
Exists: True


In [181]:
os.makedirs(output_path, exist_ok=True)

In [190]:
import os
output_path = "/content/drive/MyDrive/InkubaLM-Challenge/Output/baseline_performance/datasets"
os.makedirs(output_path, exist_ok=True)
test_csv = pd.read_csv(os.path.join(output_path, 'mt_hausa_dataset_supplement.csv'))

In [191]:
test_csv

,Unnamed: 0,ID,task,langs,data_source,instruction,inputs,targets
0,0,opus_hausa_418,mmt,eng-hau,opus100,translate the following from English into hausa.,"bring us our fathers, if what you say is true'","""sai ku zo da ubanninmu, idan kun kasance mãsu..."
1,1,opus_hausa_148,mmt,eng-hau,opus100,translate the following from English into hausa.,the first house (of worship) appointed for men...,"kuma lalle ne, ¦aki na farko da aka aza dõmin ..."
2,2,opus_hausa_233,mmt,eng-hau,opus100,translate the following from English into hausa.,so we exacted retribution from them: now see w...,"sabõda haka muka yi musu azãbar rãmuwa. to, ka..."
3,3,opus_hausa_429,mmt,eng-hau,opus100,translate the following from English into hausa.,"what, were we wearied by the first creation?","shin, to, mun kãsa ne game da halittar farko?"
4,4,ID_8f8fd13d_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from English into Hausa.,what's the symbolic meaning of the four animals?,ruhi ki ishi maa kya ab bane gi adi ki ishi maa?
...,...,...,...,...,...,...,...,...
888,888,ID_472bf3da_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from English into Hausa.,"but he answered his father, 'all these years i...","29 amma ya amsa wa mahaifinsa ya ce, 'dubi yaw..."
889,889,ID_a1bbd97d_dev_mt_eng-hau,mmt,eng-hau,wmt22,translate the following from English into Hausa.,from now on - let it go,daga - bari
890,890,opus_hausa_568,mmt,eng-hau,opus100,translate the following from English into hausa.,and there were gathered together unto solomon ...,"kuma aka tattara, dõmin sulaimãn, rundunõninsa..."
891,891,opus_hausa_143,mmt,eng-hau,opus100,translate the following from English into hausa.,"and we send the fecundating winds, then cause ...",kuma muka aika iskõki mãsu barbarar jũna sa'an...


In [184]:
test_dev.to_csv(os.path.join(output_path, 'mt_hausa_dataset_supplement.csv'))

In [179]:
test_dev.to_csv(os.path.join(output_path, 'mt_hausa_dataset_supplement.csv'), index=False)

